In [262]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import soundfile as sf
import librosa

In [264]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path(".")
TRAIN_AUDIO_DIR = BASE_DIR / "train_opus" / "audio"
TEST_AUDIO_DIR = BASE_DIR / "test_opus" / "audio"
WORD_BOUNDS_FILE = BASE_DIR / "train_opus" / "word_bounds.json"

In [266]:
with open(WORD_BOUNDS_FILE, 'r', encoding='utf-8') as f:
    phrase_bounds = json.load(f)

# Списки файлов
train_files = sorted(TRAIN_AUDIO_DIR.glob("*.opus"))
test_files = sorted(TEST_AUDIO_DIR.glob("*.opus"))

train_ids = [f.stem for f in train_files]
test_ids = [f.stem for f in test_files]

# Метки
train_labels = {audio_id: 1 if audio_id in phrase_bounds else 0 for audio_id in train_ids}

pos = sum(train_labels.values())
neg = len(train_labels) - pos
print(f"Обучающих: {len(train_ids)}")
print(f"С фразами: {pos}")
print(f"Без фраз: {neg}")
print(f"Тестовых: {len(test_ids)}")

Обучающих: 90000
С фразами: 45000
Без фраз: 45000
Тестовых: 27000


In [268]:
SAMPLE_RATE = 16000
SEGMENT_DURATION = 2.0
SEGMENT_SAMPLES = int(SAMPLE_RATE * SEGMENT_DURATION)

# Фиксированные параметры спектрограммы
N_MELS = 64
N_FFT = 512
HOP_LENGTH = 256  # Фиксируем размер временной оси

TIME_FRAMES = (SEGMENT_SAMPLES // HOP_LENGTH) + 1

In [269]:
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Устройство: {DEVICE}")

Устройство: mps


In [270]:
def read_opus_file(filepath):

    try:
        # Используем soundfile для чтения Opus
        audio, sr = sf.read(str(filepath))
        
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)
        
        if sr != SAMPLE_RATE:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
        
        return audio
    except Exception as e:
        print(f"Ошибка чтения {filepath.name}: {str(e)[:100]}")
        return np.zeros(SEGMENT_SAMPLES)

def get_fixed_segment(audio):
    """Получение фиксированного сегмента"""
    if len(audio) >= SEGMENT_SAMPLES:
        start = np.random.randint(0, len(audio) - SEGMENT_SAMPLES + 1)
        return audio[start:start + SEGMENT_SAMPLES]
    else:
        # Дополняем нулями
        return np.pad(audio, (0, SEGMENT_SAMPLES - len(audio)), mode='constant')

def create_fixed_mel_spectrogram(audio_segment):
    """Создание мел-спектрограммы"""
    # Убедимся в правильной длине
    if len(audio_segment) != SEGMENT_SAMPLES:
        audio_segment = get_fixed_segment(audio_segment)
    
    # Вычисляем мел-спектрограмму
    mel_spec = librosa.feature.melspectrogram(
        y=audio_segment,
        sr=SAMPLE_RATE,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        power=2.0
    )
    
    # Фиксируем размер по времени
    if mel_spec.shape[1] > TIME_FRAMES:
        mel_spec = mel_spec[:, :TIME_FRAMES]
    elif mel_spec.shape[1] < TIME_FRAMES:
        # Дополняем нулями справа
        pad_width = TIME_FRAMES - mel_spec.shape[1]
        mel_spec = np.pad(mel_spec, ((0, 0), (0, pad_width)), mode='constant')
    
    # Логарифмическое преобразование
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Нормализация
    if log_mel.std() > 0:
        log_mel = (log_mel - log_mel.mean()) / log_mel.std()
    
    return torch.FloatTensor(log_mel)

In [271]:
class FixedSizeAudioDataset(Dataset):
    
    def __init__(self, audio_dir, file_ids, bounds_dict, label_dict, training=True):
        self.audio_dir = audio_dir
        self.file_ids = file_ids
        self.bounds = bounds_dict
        self.labels = label_dict
        self.training = training
    
    def __len__(self):
        return len(self.file_ids)
    
    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        audio_path = self.audio_dir / f"{file_id}.opus"
        
        # Чтение аудио
        audio = read_opus_file(audio_path)
        
        # Получаем сегмент
        if self.training and self.labels[file_id] == 1 and file_id in self.bounds:
            try:
                # Для положительных примеров - вокруг фразы
                start_t, end_t = self.bounds[file_id]
                center = (start_t + end_t) / 2
                center_sample = int(center * SAMPLE_RATE)
                
                # Границы сегмента
                seg_start = max(0, center_sample - SEGMENT_SAMPLES // 2)
                seg_end = seg_start + SEGMENT_SAMPLES
                
                if seg_end <= len(audio):
                    segment = audio[seg_start:seg_end]
                else:
                    segment = get_fixed_segment(audio)
            except:
                segment = get_fixed_segment(audio)
        else:
            # Для остальных
            segment = get_fixed_segment(audio)
        
        mel_spec = create_fixed_mel_spectrogram(segment)
        
        # [1, n_mels, time_frames]
        mel_spec = mel_spec.unsqueeze(0)
        
        return mel_spec, torch.tensor(self.labels[file_id], dtype=torch.float32), file_id

In [272]:
class AudioClassifierFixed(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Полносвязные слои
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)
    
    def forward(self, x):
        # x: [batch, 1, 64, TIME_FRAMES]
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x.squeeze(-1)

In [273]:
# Разделение данных
train_idx, val_idx = train_test_split(
    train_ids,
    test_size=0.15,
    random_state=SEED,
    stratify=[train_labels[id_] for id_ in train_ids]
)

# Датасеты
train_ds = FixedSizeAudioDataset(TRAIN_AUDIO_DIR, train_idx, phrase_bounds, train_labels, training=True)
val_ds = FixedSizeAudioDataset(TRAIN_AUDIO_DIR, val_idx, phrase_bounds, train_labels, training=False)

# Проверка размерности
sample, label, fid = train_ds[0]
print(f"Размер спектрограммы: {sample.shape}")
print(f"Ожидаемый размер: [1, {N_MELS}, {TIME_FRAMES}]")

# DataLoader
BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Обучающая: {len(train_ds)}")
print(f"Валидационная: {len(val_ds)}")

# Модель
model = AudioClassifierFixed().to(DEVICE)
print(f"\nПараметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Размер спектрограммы: torch.Size([1, 64, 126])
Ожидаемый размер: [1, 64, 126]
Обучающая: 76500
Валидационная: 13500

Параметров: 101,441


In [274]:
def calculate_score_metric(y_true, y_pred_probs, threshold=0.5):
    """Вычисление метрики"""
    y_pred = (y_pred_probs >= threshold).astype(int)
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    
    frr = fn / (tp + fn) if (tp + fn) > 0 else 0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    if (1 - frr) + (1 - far) > 0:
        return 2 * (1 - frr) * (1 - far) / ((1 - frr) + (1 - far))
    return 0

@torch.no_grad()
def evaluate_model(model, loader):
    """Оценка модели"""
    model.eval()
    probs_list = []
    labels_list = []
    
    for features, labels, _ in loader:
        features = features.to(DEVICE)
        outputs = model(features)
        probs = torch.sigmoid(outputs).cpu().numpy()
        probs_list.extend(probs)
        labels_list.extend(labels.numpy())
    
    return np.array(labels_list), np.array(probs_list)

def find_threshold(labels, probs):
    """Поиск порога"""
    thresholds = np.linspace(0.1, 0.9, 17)
    best_score = 0
    best_thresh = 0.5
    
    for thresh in thresholds:
        score = calculate_score_metric(labels, probs, thresh)
        if score > best_score:
            best_score = score
            best_thresh = thresh
    
    return best_score, best_thresh

In [279]:
# Настройка
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 5
best_val_score = 0
best_state = None
best_thresh = 0.5


for epoch in range(EPOCHS):
    # Обучение
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_x, batch_y, _ in pbar:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / len(train_loader)
    
    # Валидация
    val_labels, val_probs = evaluate_model(model, val_loader)
    val_score, curr_thresh = find_threshold(val_labels, val_probs)
    
    # LR schedule
    scheduler.step()
    
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Score={val_score:.4f}, Thr={curr_thresh:.3f}")
    
    # Сохранение
    if val_score > best_val_score:
        best_val_score = val_score
        best_thresh = curr_thresh
        best_state = model.state_dict().copy()
        print(f"Лучшая модель!!!")

print(f"\nОбучение завершено")
print(f"Лучший score: {best_val_score:.4f}")
print(f"Порог: {best_thresh:.3f}")

# Загрузка лучшей модели
model.load_state_dict(best_state)

Epoch 1/5: 100%|██████████| 598/598 [20:04<00:00,  2.01s/it, loss=0.589]


Epoch 1: Loss=0.5404, Score=0.6664, Thr=0.400
Лучшая модель!!!


Epoch 2/5: 100%|██████████| 598/598 [19:38<00:00,  1.97s/it, loss=0.597]


Epoch 2: Loss=0.5143, Score=0.6858, Thr=0.550
Лучшая модель!!!


Epoch 3/5: 100%|██████████| 598/598 [19:44<00:00,  1.98s/it, loss=0.487]


...

Epoch 4/5: 100%|██████████| 598/598 [19:48<00:00,  1.99s/it, loss=0.509]


Epoch 4: Loss=0.4762, Score=0.6903, Thr=0.800
Лучшая модель!!!


Epoch 5/5: 100%|██████████| 598/598 [19:47<00:00,  1.99s/it, loss=0.406]


Epoch 5: Loss=0.4644, Score=0.7023, Thr=0.150
Лучшая модель!!!

Обучение завершено
Лучший score: 0.7023
Порог: 0.150


<All keys matched successfully>

In [280]:
@torch.no_grad()
def predict_audio(model, filepath):
    """Предсказание для файла"""
    model.eval()
    
    # Чтение
    audio = read_opus_file(filepath)
    
    # Сегменты
    segments = []
    
    # Центральный
    if len(audio) >= SEGMENT_SAMPLES:
        center = len(audio) // 2
        start = max(0, center - SEGMENT_SAMPLES // 2)
        segments.append(audio[start:start + SEGMENT_SAMPLES])
    else:
        segments.append(audio)
    
    # Начальный и конечный
    if len(audio) > SEGMENT_SAMPLES * 1.5:
        segments.append(audio[:SEGMENT_SAMPLES])
        segments.append(audio[-SEGMENT_SAMPLES:])
    
    # Предсказания
    probs = []
    for seg in segments:
        mel = create_fixed_mel_spectrogram(seg).unsqueeze(0).unsqueeze(0).to(DEVICE)
        prob = torch.sigmoid(model(mel)).item()
        probs.append(prob)
    
    return max(probs) if probs else 0.0

# Прогнозирование тестовых
print("===Прогнозирование тестовых файлов===")
test_probs = []

for file in tqdm(test_files, desc="Обработка"):
    prob = predict_audio(model, file)
    test_probs.append(prob)

# Порог
test_preds = [1 if p >= best_thresh else 0 for p in test_probs]

print(f"\nРезультаты:")
print(f"Всего: {len(test_preds)}")
print(f"Положительных: {sum(test_preds)}")
print(f"Отрицательных: {len(test_preds) - sum(test_preds)}")

===Прогнозирование тестовых файлов===


Обработка: 100%|██████████| 27000/27000 [16:25<00:00, 27.39it/s]   


Результаты:
Всего: 27000
Положительных: 18968
Отрицательных: 8032


In [281]:
# CSV файл
submission = pd.DataFrame({
    'id': test_ids,
    'label': test_preds
})

submission.to_csv("submition.csv", index=False)

print(submission.head())
print(f"\nРаспределение: {submission['label'].value_counts().to_dict()}")

                                         id  label
0  0000219778122723066859323624505982384475      0
1  0000920560142346477464477964040846645823      1
2  0002106775361063830068199242310438122126      1
3  0002161736146841817059430282255903999813      0
4  0002303832386140303186933286284938192307      0

Распределение: {1: 18968, 0: 8032}
